# LLMSTU full run — SHARD BY SHARD (streamed to HF)
Processes one shard at a time: **download → crop → dedup → caption → upload crops+labels
→ mark done → free disk**, then repeats. Resumable across VM wipes: shards already
marked done on the Hub are skipped. After all shards finish, build the golden set.

Runs Qwen3.5 via **transformers + sdpa** (vLLM needs CUDA 13; Colab is CUDA 12).
Throughput comes from big batches on the ~95GB GPU + hf_transfer. Only ~1 shard of
data is on disk at any time.

## 1. Setup

In [ ]:
REPO = "vaelkokach/LLMSTU-pipeline"
import os
try:
    from google.colab import userdata
    def _secret(n):
        try: return userdata.get(n)
        except Exception: return None
except Exception:
    def _secret(n): return None
if REPO and not os.path.isdir("/content/LLMSTU-pipeline"):
    gh=_secret("GITHUB_TOKEN"); auth=f"{gh}@" if gh else ""
    !git clone https://{auth}github.com/{REPO}.git /content/LLMSTU-pipeline
if os.path.isdir("/content/LLMSTU-pipeline"): %cd /content/LLMSTU-pipeline
# transformers + sdpa path (no vLLM/flash-attn needed on Colab). hf_transfer speeds HF I/O.
!pip install -q "ultralytics>=8.3.0" "transformers>=4.57.0" accelerate hf_transfer \
    "huggingface_hub>=0.35.0" pandas pyarrow pyyaml pillow numpy
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = _secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Set HF_TOKEN in Colab Secrets"
from huggingface_hub import login; login(os.environ["HF_TOKEN"])
import torch; print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Steps 1–5 per shard (the whole loop)
Loads the caption model ONCE (transformers, bf16, sdpa, batch 48), then streams every
shard to `WaelK/LLMSTU-crops`. Re-run this cell after any disconnect — it skips shards
already done. Watch shard_000 land on HF, then let it ride.

In [ ]:
!HF_HUB_ENABLE_HF_TRANSFER=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/12_run_shards.py --repo WaelK/LLMSTU-crops --num-shards 21

#### (optional) run specific shards only
Handy to test on one shard first, or to parallelise across sessions.

In [ ]:
# !python scripts/12_run_shards.py --repo WaelK/LLMSTU-crops --shards shard_000

## 3. Step 6 — build the golden set (after ALL shards done)
Gathers labels from HF, samples (stratified + hard), downloads only the sampled crops.

In [ ]:
!python scripts/14_golden_from_hf.py --repo WaelK/LLMSTU-crops --n 1500 \
    --stratify-fields engagement_level,activity,gaze_direction --min-per-class 40 --hard-frac 0.2
!cd labeling && zip -qr /content/labeling_full.zip index.html data.json
from google.colab import files; files.download("/content/labeling_full.zip")

## 4. Step 7 — hand-label + score
Unzip, open `index.html`, label, Export `golden.json`, upload it back below, then score.

In [ ]:
from google.colab import files
up = files.upload()   # your golden.json
import shutil; shutil.move(list(up)[0], "labeling/golden.json"); print("saved labeling/golden.json")

In [ ]:
!python scripts/07_eval_golden.py --golden labeling/golden.json
from IPython.display import HTML
HTML(open("work/eval/golden_report.html").read())

## 5. Step 8 — finalize
Builds `crops/metadata.jsonl` (image+caption join) + `pseudo_labels.parquet` and pushes
the golden set. Crops themselves were already uploaded per shard.

In [ ]:
!python scripts/13_finalize.py --repo WaelK/LLMSTU-crops --golden labeling/golden.json